## Cohort Table Verification

Counts scans, samples, and TEA cases (T11/T13) per split.

**Approach: mirrors `dataloader_VerSe.py` + positional Rule A.**

- File collection: `get_filtered_files_across_dsnames` with `extra_sacral_verts=frozenset()`, same as training
- Scan/sample counts from actual disk files
- Only PIDs in `pid_corrections.json` are corrected (whitelist, same as dataloader)
- **Rule A (T13)**: vert28 appears **positionally between** vert19 and vert20 in `vert_label` list from Excel  
  → 28→20, v≥20 (v≠28) → v+1; shifted patches = files where disk vert ≥ 20
- **Rule B (T11)**: vert19 absent, vert18 and vert20 present (set check)  
  → v≥20 → v−1; shifted patches = files where disk vert ≥ 20

In [8]:
import sys
import json
import re
from pathlib import Path
from collections import defaultdict

# make project root importable
sys.path.insert(0, str(Path("/home/student/lisa_ma")))
import datasets.VerSe.get_data_VerSe as get_data_VerSe

ROOT       = Path("/home/student/lisa_ma/prepared")
EXCEL_PATH = "/home/student/lisa_ma/datasets/VerSe/data_filter_joined.xlsx"
PID_JSON   = "/home/student/lisa_ma/datasets/VerSe/pid_corrections.json"

COMMON_KWARGS = dict(
    root_dir=ROOT,
    excel_path=EXCEL_PATH,
    glob_pattern="*.npz",
    pid_col="pid",
    dsname_col="dsname",
    verts_col="vert_label",
    check_complete=False,          # skip completeness check (not needed here)
    apply_excel_filter=True,
    extra_sacral_verts=frozenset(),  # same as training — no extra sacral
)

split_files = {}
for split in ["train", "val", "test"]:
    files, _ = get_data_VerSe.get_filtered_files_across_dsnames(split=split, **COMMON_KWARGS)
    split_files[split] = files
    print(f"{split}: {len(files)} files")

train: 19266 files
val: 3736 files
test: 2211 files


## 1. Scan and sample counts

In [9]:
def get_pid(path):
    """Mirrors VerSeDataset.get_pid_from_path."""
    m = re.search(r'_sub-([^.]+)', str(path))
    return m.group(1) if m else "unknown"

def get_vert(path):
    """Mirrors VerSeDataset.get_vert_number."""
    m = re.search(r'_vert(\d+)', str(path))
    return int(m.group(1)) if m else -1

scan_sample_counts = {}
split_pid_to_verts = {}  # disk vert sets per PID per split

for split in ["train", "val", "test"]:
    files = split_files[split]
    pids = set(get_pid(f) for f in files)
    pid_verts = defaultdict(set)
    for f in files:
        pid = get_pid(f)
        v = get_vert(f)
        if v >= 0:
            pid_verts[pid].add(v)
    split_pid_to_verts[split] = dict(pid_verts)
    scan_sample_counts[split] = {"scans": len(pids), "samples": len(files)}

print(f"{'Split':8s} {'Scans':>8} {'Samples':>10}")
print("-" * 30)
total_scans = total_samples = 0
for split in ["train", "val", "test"]:
    c = scan_sample_counts[split]
    print(f"{split:8s} {c['scans']:>8} {c['samples']:>10}")
    total_scans += c['scans']
    total_samples += c['samples']
print("-" * 30)
print(f"{'total':8s} {total_scans:>8} {total_samples:>10}")

Split       Scans    Samples
------------------------------
train        1176      19266
val           229       3736
test          136       2211
------------------------------
total        1541      25213


## 2. Load pid_corrections.json whitelist

In [10]:
with open(PID_JSON) as f:
    pid_data = json.load(f)

pid_fix_1820 = set(pid_data.get("pid_fix_1820", []))  # Rule B candidates
pid_fix_28   = set(pid_data.get("pid_fix_28",   []))  # Rule A candidates
whitelist    = pid_fix_1820 | pid_fix_28

print(f"Rule B whitelist ({len(pid_fix_1820)} PIDs): {sorted(pid_fix_1820)}")
print(f"Rule A whitelist ({len(pid_fix_28)} PIDs):   {sorted(pid_fix_28)}")

Rule B whitelist (12 PIDs): ['ID0049_ses-14082007_ce-pv', 'ID0067_ses-04062009_ce-pv', 'ID0297_ses-29102010_ce-pv', 'ID0387_ses-25062010_ce-pv', 'ID0423_ses-17102007_ce-pv', 'ID0423_ses-28072008_ce-pv', 'verse559', 'verse606', 'verse626_dir-iso', 'verse636', 'verse642_dir-sag', 'verse814']
Rule A whitelist (7 PIDs):   ['ctfu00610_ses-20091111_ce-pv', 'ctfu00897_ses-20170303_ce-art', 'ctfu01006_ses-20160915_ce-no', 'verse582_dir-iso', 'verse588', 'verse601', 'verse631']


## 3. Apply Rule A / Rule B (set-based, same logic as `_compute_pid_corrections`)

The dataloader computes corrections over the **train+val union** to avoid split-dependent differences.  
We replicate that, then count corrections per split separately.

In [11]:
import pandas as pd
import ast

def compute_corrections(file_paths, target_pids, excel_path):
    """
    Mirrors VerSeDataLoader._compute_pid_corrections, but uses positional Rule A.
    - Vert sets built from disk filenames (for scan membership and shifted-patch counts)
    - Ordered vert_label from Excel used for positional Rule A check
    - Only PIDs in target_pids are processed (whitelist from pid_corrections.json)
    Returns: dict pid -> 'A' | 'B' | None
    """
    # Build disk vert sets for whitelisted PIDs
    pid_to_verts = defaultdict(set)
    for p in file_paths:
        pid = get_pid(p)
        if pid not in target_pids:
            continue
        v = get_vert(p)
        if v >= 0:
            pid_to_verts[pid].add(v)

    # Load ordered vert_label from Excel for positional Rule A check
    df_ex = pd.read_excel(excel_path)
    pid_to_vl_ordered = {}
    for _, row in df_ex.iterrows():
        pid = str(row["pid"]).strip()
        if pid not in target_pids:
            continue
        vl = row["vert_label"]
        if isinstance(vl, str):
            pid_to_vl_ordered[pid] = [int(v) for v in ast.literal_eval(vl)]
        else:
            pid_to_vl_ordered[pid] = [int(v) for v in vl]

    result = {}
    for pid, verts in pid_to_verts.items():
        vl_ordered = pid_to_vl_ordered.get(pid, [])
        vl_set = set(vl_ordered)

        # Rule A: positional — vert28 must appear between vert19 and vert20 in vert_label
        rule_a = False
        if 28 in verts and 20 in verts and 28 in vl_set and 19 in vl_set and 20 in vl_set:
            try:
                if vl_ordered.index(19) < vl_ordered.index(28) < vl_ordered.index(20):
                    rule_a = True
            except ValueError:
                pass

        if rule_a:
            result[pid] = 'A'
        # Rule B: set-based — vert19 absent, vert18 and vert20 present on disk
        elif 20 in verts and 19 not in verts and 18 in verts:
            result[pid] = 'B'
        else:
            result[pid] = None
    return result

# Dataloader computes corrections over train+val union
train_val_files = split_files["train"] + split_files["val"]
corrections_tv = compute_corrections(train_val_files, whitelist, EXCEL_PATH)

# Test corrections applied independently (not part of training dataloader)
corrections_test = compute_corrections(split_files["test"], whitelist, EXCEL_PATH)

print("Train+val corrections:")
for pid, rule in sorted(corrections_tv.items()):
    if rule:
        print(f"  Rule {rule}: {pid}")
print("\nTest corrections:")
for pid, rule in sorted(corrections_test.items()):
    if rule:
        print(f"  Rule {rule}: {pid}")

Train+val corrections:
  Rule B: ID0049_ses-14082007_ce-pv
  Rule B: ID0067_ses-04062009_ce-pv
  Rule B: ID0297_ses-29102010_ce-pv
  Rule B: ID0387_ses-25062010_ce-pv
  Rule B: ID0423_ses-17102007_ce-pv
  Rule B: ID0423_ses-28072008_ce-pv
  Rule A: ctfu00610_ses-20091111_ce-pv
  Rule A: ctfu00897_ses-20170303_ce-art
  Rule A: ctfu01006_ses-20160915_ce-no
  Rule B: verse559
  Rule A: verse582_dir-iso
  Rule A: verse588
  Rule A: verse601
  Rule B: verse606
  Rule B: verse626_dir-iso
  Rule A: verse631
  Rule B: verse636

Test corrections:
  Rule B: verse642_dir-sag
  Rule B: verse814


## 4. Count TEA scans and shifted patches per split

In [12]:
def count_for_split(split, corrections):
    """Count TEA scans and shifted patches for a given split using a corrections dict."""
    files = split_files[split]
    pid_verts = split_pid_to_verts[split]

    rule_a_pids, rule_b_pids = [], []
    shifted_a = shifted_b = 0

    for pid, rule in corrections.items():
        verts = pid_verts.get(pid, set())
        if not verts:
            continue  # PID not in this split
        n_shifted = sum(1 for v in verts if v >= 20)
        if rule == 'A':
            rule_a_pids.append(pid)
            shifted_a += n_shifted
        elif rule == 'B':
            rule_b_pids.append(pid)
            shifted_b += n_shifted

    return {
        'rule_a_pids': sorted(rule_a_pids),
        'rule_b_pids': sorted(rule_b_pids),
        'rule_a_scans': len(rule_a_pids),
        'rule_b_scans': len(rule_b_pids),
        'shifted_a': shifted_a,
        'shifted_b': shifted_b,
    }

results = {}
for split in ["train", "val"]:
    results[split] = count_for_split(split, corrections_tv)
results["test"] = count_for_split("test", corrections_test)

print(f"{'':32s} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 68)

for key, label in [("scans", "Scans"), ("samples", "Samples")]:
    vals = [scan_sample_counts[s][key] for s in ["train", "val", "test"]]
    print(f"{label:32s} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {sum(vals):>8}")

print()

for rule_key, label in [("rule_a_scans", "TEA-T13 scans (Rule A)"), ("rule_b_scans", "TEA-T11 scans (Rule B)")]:
    vals = [results[s][rule_key] for s in ["train", "val", "test"]]
    print(f"{label:32s} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {sum(vals):>8}")

tea_vals = [results[s]['rule_a_scans'] + results[s]['rule_b_scans'] for s in ["train", "val", "test"]]
print(f"{'TEA total scans':32s} {tea_vals[0]:>8} {tea_vals[1]:>8} {tea_vals[2]:>8} {sum(tea_vals):>8}")

rate_vals = [round(100 * tea_vals[i] / scan_sample_counts[s]['scans'], 1)
             for i, s in enumerate(["train", "val", "test"])]
print(f"{'TEA rate (%)':32s} {rate_vals[0]:>8} {rate_vals[1]:>8} {rate_vals[2]:>8} {round(100*sum(tea_vals)/total_scans,1):>8}")

print()

for sh_key, label in [("shifted_a", "Patches shifted Rule A (T13)"), ("shifted_b", "Patches shifted Rule B (T11)")]:
    vals = [results[s][sh_key] for s in ["train", "val", "test"]]
    print(f"{label:32s} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {sum(vals):>8}")

total_sh = [results[s]['shifted_a'] + results[s]['shifted_b'] for s in ["train", "val", "test"]]
print(f"{'Total patches shifted':32s} {total_sh[0]:>8} {total_sh[1]:>8} {total_sh[2]:>8} {sum(total_sh):>8}")

                                    Train      Val     Test    Total
--------------------------------------------------------------------
Scans                                1176      229      136     1541
Samples                             19266     3736     2211    25213

TEA-T13 scans (Rule A)                  6        1        0        7
TEA-T11 scans (Rule B)                  8        2        2       12
TEA total scans                        14        3        2       19
TEA rate (%)                          1.2      1.3      1.5      1.2

Patches shifted Rule A (T13)           33        6        0       39
Patches shifted Rule B (T11)           44       11       11       66
Total patches shifted                  77       17       11      105


## 5. Detail: per-PID correction cases

In [13]:
for split in ["train", "val", "test"]:
    r = results[split]
    print(f"\n=== {split.upper()} ===")
    print(f"  Rule A (T13, {r['rule_a_scans']} scans, {r['shifted_a']} patches shifted):")
    for pid in r['rule_a_pids']:
        verts = sorted(split_pid_to_verts[split].get(pid, set()))
        n = sum(1 for v in verts if v >= 20)
        print(f"    {pid:55s} disk_verts={verts}  shifted={n}")
    print(f"  Rule B (T11, {r['rule_b_scans']} scans, {r['shifted_b']} patches shifted):")
    for pid in r['rule_b_pids']:
        verts = sorted(split_pid_to_verts[split].get(pid, set()))
        n = sum(1 for v in verts if v >= 20)
        print(f"    {pid:55s} disk_verts={verts}  shifted={n}")


=== TRAIN ===
  Rule A (T13, 6 scans, 33 patches shifted):
    ctfu00610_ses-20091111_ce-pv                            disk_verts=[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 28]  shifted=5
    ctfu00897_ses-20170303_ce-art                           disk_verts=[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 28]  shifted=4
    ctfu01006_ses-20160915_ce-no                            disk_verts=[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28]  shifted=6
    verse588                                                disk_verts=[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28]  shifted=6
    verse601                                                disk_verts=[8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28]  shifted=6
    verse631                                                disk_verts=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 28]  shifted=

## 6. No-whitelist scan: all samples, positional Rule A

Check whether the json whitelist misses any TEA cases.  
- Source: `vert_label` ordered lists from Excel (positional check possible)  
- Rule A: vert28 must appear **between** vert19 and vert20 in the list  
- Rule B: vert19 absent, vert18 and vert20 present  
- No whitelist — all 1,541 filtered scans are checked

In [14]:
import pandas as pd
import ast

EXCEL_PATH = "/home/student/lisa_ma/datasets/VerSe/data_filter_joined.xlsx"
df_main = pd.read_excel(EXCEL_PATH)

def parse_verts_ordered(vl):
    if isinstance(vl, str):
        return [int(v) for v in ast.literal_eval(vl)]
    return [int(v) for v in vl]

def detect_rule_positional(vl_list):
    """
    Rule A: vert28 positionally between vert19 and vert20 in the ordered vert_label list.
    Rule B: vert19 absent, vert18 and vert20 present (set check, order doesn't matter).
    Returns ('A', n_shifted), ('B', n_shifted), or (None, 0).
    n_shifted = patches with disk vert >= 20 (those that get a new label).
    """
    vl_set = set(vl_list)
    # Rule A: positional
    if 28 in vl_set and 19 in vl_set and 20 in vl_set:
        try:
            if vl_list.index(19) < vl_list.index(28) < vl_list.index(20):
                return 'A', sum(1 for v in vl_list if v >= 20)
        except ValueError:
            pass
    # Rule B: set-based
    if 19 not in vl_set and 18 in vl_set and 20 in vl_set:
        return 'B', sum(1 for v in vl_list if v >= 20)
    return None, 0

# Apply to all filtered scans
df_main["vl_list"] = df_main["vert_label"].apply(parse_verts_ordered)
df_main["n_verts"]  = df_main["vl_list"].apply(len)

rows = []
for split in ["train", "val", "test"]:
    sub = df_main[
        (df_main["split"] == split) &
        (df_main["has_T1"] == 1) &
        (df_main["is_consecutive"] == 1)
    ]
    for _, row in sub.iterrows():
        rule, n_shifted = detect_rule_positional(row["vl_list"])
        rows.append({"split": split, "pid": row["pid"], "dsname": row["dsname"],
                     "rule": rule, "n_shifted": n_shifted})

df_all = pd.DataFrame(rows)

# Summary table
print(f"{'':32s} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 68)
for rule, label in [('A', 'TEA-T13 scans (Rule A, positional)'),
                    ('B', 'TEA-T11 scans (Rule B, no whitelist)')]:
    vals = [int((df_all[df_all['split']==s]['rule'] == rule).sum()) for s in ["train","val","test"]]
    print(f"{label:32s} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {sum(vals):>8}")

tea_vals = [int(df_all[df_all['split']==s]['rule'].isin(['A','B']).sum()) for s in ["train","val","test"]]
print(f"{'TEA total':32s} {tea_vals[0]:>8} {tea_vals[1]:>8} {tea_vals[2]:>8} {sum(tea_vals):>8}")
for rule, sh_label in [('A','Patches shifted Rule A'),('B','Patches shifted Rule B')]:
    vals = [int(df_all[(df_all['split']==s)&(df_all['rule']==rule)]['n_shifted'].sum()) for s in ["train","val","test"]]
    print(f"{sh_label:32s} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8} {sum(vals):>8}")

print()
# Show any NEW cases not in the json whitelist
print("Cases NOT in pid_corrections.json whitelist:")
new_cases = df_all[df_all['rule'].notna() & ~df_all['pid'].isin(whitelist)]
if new_cases.empty:
    print("  None — whitelist is complete.")
else:
    for _, r in new_cases.iterrows():
        print(f"  Rule {r['rule']} | {r['split']:5s} | {r['pid']}  shifted={r['n_shifted']}")

print()
# Show any whitelist PIDs where rule did NOT fire (rule is None)
print("Whitelist PIDs where no rule fired (rule=None):")
for pid in whitelist:
    row = df_all[df_all['pid'] == pid]
    if row.empty:
        print(f"  {pid}  — not found in filtered set")
    elif row.iloc[0]['rule'] is None:
        print(f"  {pid}  — rule=None, vl={row.iloc[0]['rule']}")

                                    Train      Val     Test    Total
--------------------------------------------------------------------
TEA-T13 scans (Rule A, positional)        6        1        0        7
TEA-T11 scans (Rule B, no whitelist)        8        2        2       12
TEA total                              14        3        2       19
Patches shifted Rule A                 33        6        0       39
Patches shifted Rule B                 44       11       11       66

Cases NOT in pid_corrections.json whitelist:
  None — whitelist is complete.

Whitelist PIDs where no rule fired (rule=None):


## 7. Mislabeled patches in training — with training sample_ids

Identifies every patch whose **training class ≠ GT class**, keyed by `{pid}_vert{training_vert}`
(the same id the model and GLARE see).

**Priority / logic:**
1. **Rule A (pid_fix_28)** — T13 with vert28 on disk, correction applied:  
   - disk vert28 → training vert20 = Lumbar, GT = Thoracic  
   - disk vert24 → training vert25 = Sacral, GT = Lumbar (only if vert24 on disk)
2. **Rule B (pid_fix_1820)** — T11, vert19 missing on disk, correction applied:  
   - disk vert20 → training vert19 = Thoracic, GT = Lumbar  
   - disk vert25 → training vert24 = Lumbar, GT = Sacral (only if vert25 on disk)
3. **T13_excel** — T13 annotation in anomaly Excel, no correction (not in any whitelist):  
   - disk vert20 = training vert20 = Lumbar, GT = Thoracic  
   - disk vert25 = training vert25 = Sacral, GT = Lumbar (only if vert25 on disk)

Rule A/B take precedence — a PID in both the anomaly Excel and a whitelist uses only Rule A/B.

In [ ]:
import pandas as pd
import ast

ANOM_PATH = "/home/student/lisa_ma/evaluation/A_CT_ANOMALY_LABELv9.xlsx"
df_anom = pd.read_excel(ANOM_PATH)

# T13-annotated PIDs from anomaly Excel (fid column matches pid in data_filter_joined)
t13_pids = set(df_anom[df_anom['T13'] == 1]['fid'].str.strip())

mislabels = []

for split in ['train', 'val']:
    pid_verts = split_pid_to_verts[split]  # disk vert sets from actual loaded files
    corrections = corrections_tv          # same correction map for train+val

    # ---- Rule A (T13 with vert28, correction applied) ----
    for pid in sorted(pid_fix_28):
        if pid not in pid_verts or corrections.get(pid) != 'A':
            continue
        disk_verts = pid_verts[pid]
        # disk vert28 → training vert20: Lumbar in training, Thoracic in GT
        mislabels.append({
            'sample_id': f'{pid}_vert20', 'pid': pid, 'split': split,
            'training_vert': 20, 'disk_vert': 28,
            'training_class': 'Lumbar', 'true_class': 'Thoracic',
            'anomaly_type': 'T13_extra', 'source': 'Rule A',
        })
        # disk vert24 → training vert25: Sacral in training, Lumbar in GT
        if 24 in disk_verts:
            mislabels.append({
                'sample_id': f'{pid}_vert25', 'pid': pid, 'split': split,
                'training_vert': 25, 'disk_vert': 24,
                'training_class': 'Sacral', 'true_class': 'Lumbar',
                'anomaly_type': 'T13_extra', 'source': 'Rule A',
            })

    # ---- Rule B (T11, vert19 missing, correction applied) ----
    for pid in sorted(pid_fix_1820):
        if pid not in pid_verts or corrections.get(pid) != 'B':
            continue
        disk_verts = pid_verts[pid]
        # disk vert20 → training vert19: Thoracic in training, Lumbar in GT
        mislabels.append({
            'sample_id': f'{pid}_vert19', 'pid': pid, 'split': split,
            'training_vert': 19, 'disk_vert': 20,
            'training_class': 'Thoracic', 'true_class': 'Lumbar',
            'anomaly_type': 'T12_missing', 'source': 'Rule B',
        })
        # disk vert25 → training vert24: Lumbar in training, Sacral in GT
        if 25 in disk_verts:
            mislabels.append({
                'sample_id': f'{pid}_vert24', 'pid': pid, 'split': split,
                'training_vert': 24, 'disk_vert': 25,
                'training_class': 'Lumbar', 'true_class': 'Sacral',
                'anomaly_type': 'T12_missing', 'source': 'Rule B',
            })

    # ---- T13_excel (T13 from anomaly Excel, no correction) ----
    for pid in sorted(t13_pids - whitelist):
        if pid not in pid_verts:
            continue
        disk_verts = pid_verts[pid]
        # disk vert20 = training vert20: Lumbar in training, Thoracic in GT
        if 20 in disk_verts:
            mislabels.append({
                'sample_id': f'{pid}_vert20', 'pid': pid, 'split': split,
                'training_vert': 20, 'disk_vert': 20,
                'training_class': 'Lumbar', 'true_class': 'Thoracic',
                'anomaly_type': 'T13_extra', 'source': 'T13_excel',
            })
        # disk vert25 = training vert25: Sacral in training, Lumbar in GT
        if 25 in disk_verts:
            mislabels.append({
                'sample_id': f'{pid}_vert25', 'pid': pid, 'split': split,
                'training_vert': 25, 'disk_vert': 25,
                'training_class': 'Sacral', 'true_class': 'Lumbar',
                'anomaly_type': 'T13_extra', 'source': 'T13_excel',
            })

df_mislabels = pd.DataFrame(mislabels)

# Summary
print("=== Mislabeled patches per split ===")
summary = df_mislabels.groupby(['split', 'anomaly_type', 'training_class', 'true_class']).size().reset_index(name='patches')
print(summary.to_string())
print()
print(f"Total train: {(df_mislabels['split']=='train').sum()}")
print(f"Total val:   {(df_mislabels['split']=='val').sum()}")
print()

# Cross-check train against GT file
GT_PATH = "/home/student/lisa_ma/evaluation/gt_mislabels_complete_LO.xlsx"
df_gt = pd.read_excel(GT_PATH)
# GT file uses sample_id = {pid}_vert{training_vert}; filter to PIDs in current corrections
invalid_pids = {'ID0102_ses-21052008_ce-pv', 'liver0106'}  # not in current pid_fix_1820
df_gt_current = df_gt[~df_gt['pid'].isin(invalid_pids)]
gt_ids = set(df_gt_current['sample_id'])
our_train_ids = set(df_mislabels[df_mislabels['split']=='train']['sample_id'])
only_in_gt  = gt_ids - our_train_ids
only_in_ours = our_train_ids - gt_ids
print(f"Cross-check vs GT file (train, {len(df_gt_current)} entries after removing non-current PIDs):")
print(f"  In GT file but not our list: {sorted(only_in_gt) if only_in_gt else 'none'}")
print(f"  In our list but not GT file: {sorted(only_in_ours) if only_in_ours else 'none'}")
print()
print("Full mislabel list (train):")
print(df_mislabels[df_mislabels['split']=='train'][
    ['sample_id','training_class','true_class','anomaly_type','source']
].sort_values('sample_id').to_string(index=False))
print()
print("Full mislabel list (val):")
print(df_mislabels[df_mislabels['split']=='val'][
    ['sample_id','training_class','true_class','anomaly_type','source']
].sort_values('sample_id').to_string(index=False))